In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
import re
import glob 

On associe une province à chaque Ping en bouclant sur les fichier provinces

In [6]:
PATH = Path("/data/rd_exchange/salbernhe/provinces/netcdf_monthly_provinces/provinces_Y1998_M01.nc")
ds = xr.open_dataset(PATH)
ds

<xarray.Dataset> Size: 35MB
Dimensions:    (latitude: 2040, longitude: 4320)
Coordinates:
  * latitude   (latitude) float32 8kB -80.0 -79.92 -79.83 ... 89.75 89.83 89.92
  * longitude  (longitude) float32 17kB -180.0 -179.9 -179.8 ... 179.8 179.9
Data variables:
    province   (latitude, longitude) float32 35MB ...
Attributes:
    title:        Regions biome T, Str, npp (from GLORYS/VGPM)
    institution:  CLS
    references:   http://www.cls.fr

In [8]:
ds_sv_path = Path("/data/rd_exchange/salbernhe/data/acoustics/profiles38kHz10t0750m20251006.nc")


ds_sv = xr.open_dataset(ds_sv_path)
ds_sv = ds_sv.assign_coords(
    year=("time", ds_sv["time"].dt.year.values),
    month=("time", ds_sv["time"].dt.month.values)

)

idx_ping_prov = np.full(ds_sv.dims["time"], np.nan, dtype=object)

prov_folder = sorted(glob.glob("/data/rd_exchange/salbernhe/provinces/netcdf_monthly_provinces/provinces_Y*_M*.nc"))

/tmp/ipykernel_2796050/3501772755.py:11: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  idx_ping_prov = np.full(ds_sv.dims["time"], np.nan, dtype=object)


In [11]:
#1. On boucle sur les fichies provinces

for file_prov in prov_folder:
    name = file_prov.split("/")[-1].replace(".nc", "")   # "provinces_Y1998_M01"

    year_str = int(name.split("_")[1].replace("Y", ""))
    month_str = int(name.split("_")[2].replace("M", ""))

    mask= (ds_sv["year"]==year_str) & (ds_sv["month"]==month_str)
    if not mask.any():
        continue
   
    # on va chercher le point de grille le plus proche 

    sub = ds_sv.sel(time=mask) #on applique le mask qui filtre le ds_sv et ne garde que les ping du mois/année en cours
    ds_prov = xr.open_dataset(file_prov) #on ouvre le fichier de province correspondant
    prov_sel = ds_prov["province"].sel(latitude=sub["latitude"], longitude=sub["longitude"], method="nearest") #on récupère le fameux point de grille

    idx_ping_prov[mask.values] = prov_sel.values #on ajoute au tableau la province correspondant aux coordonnées pour le mois considéré
    ds_prov.close()
    

ds_sv = ds_sv.assign_coords(province=("time", idx_ping_prov)) #on ajoute la province au dataset acoustique

    # --- Vérification rapide ---
n_missing = np.sum(idx_ping_prov == None) if idx_ping_prov.dtype == object else np.isnan(idx_ping_prov.astype(float)).sum()
print(f"Pings sans province assignée : {n_missing} / {ds_sv.dims['time']}")
print("Provinces uniques trouvées :", np.unique([p for p in idx_ping_prov if p is not None]))







Pings sans province assignée : 0 / 541061
Provinces uniques trouvées : [101. 102. 103. 201. 202. 203. 204. 205. 301. 302. 303. 304. 306. 307.
 401. 402. 403. 405. 406. 502. 503. 601.  nan]


/tmp/ipykernel_2796050/1475584839.py:27: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"Pings sans province assignée : {n_missing} / {ds_sv.dims['time']}")


In [13]:
ds_sv




<xarray.Dataset> Size: 364MB
Dimensions:     (file_index: 318, time: 541061, range: 75)
Coordinates:
  * file_index  (file_index) int64 3kB 1 2 3 4 5 6 7 ... 313 314 315 316 317 318
  * time        (time) datetime64[ns] 4MB 2008-04-09T10:00:34.000003072 ... 2...
    year        (time) int64 4MB 2008 2008 2008 2008 ... 2006 2006 2006 2006
    month       (time) int64 4MB 4 4 4 4 4 4 4 4 4 4 4 ... 3 3 3 3 3 3 3 3 3 3 3
    province    (time) object 4MB 601.0 601.0 601.0 601.0 ... 102.0 102.0 102.0
  * range       (range) int64 600B 10 20 30 40 50 60 ... 700 710 720 730 740 750
Data variables:
    file_label  (file_index) <U116 148kB ...
    longitude   (time) float64 4MB ...
    latitude    (time) float64 4MB ...
    file        (time) int64 4MB ...
    interval    (time) float64 4MB ...
    sunangle    (time) float64 4MB ...
    Sv          (range, time) float64 325MB ...
    frequency   |S1 1B ...

on converti les sv en linéaire et on fait la moyenne groupée

In [18]:
def db_to_lin(x):
    return 10.0 ** (x / 10.0)

def lin_to_db(x):
    return 10.0 * np.log10(x)

sv_lin= db_to_lin(ds_sv["Sv"])#passage en linéaire

sv_lin_moy = sv_lin.groupby(ds_sv["province"]).map(lambda x: x.groupby(x["month"]).mean(dim="time", skipna=True)) # calcul de la moyenne (map->opération pour toutes les prov séparées, recollées par xarray)

sv_mean_db = lin_to_db(sv_lin_moy)#on repasse en db


In [19]:
sv_mean_db

<xarray.DataArray 'Sv' (range: 75, province: 22, month: 12)> Size: 158kB
array([[[-71.76235049, -69.81583778, -69.90639183, ..., -71.84287139,
         -78.93563011, -69.37688326],
        [         nan,          nan, -67.30834869, ...,          nan,
         -68.70818586, -69.11633416],
        [-68.81839064, -69.44036779, -68.09978245, ...,          nan,
         -73.98989596, -68.48588009],
        ...,
        [         nan,          nan, -63.00236591, ...,          nan,
                  nan,          nan],
        [-75.68594803, -78.61502446, -79.75571748, ..., -70.96218129,
         -72.73609574, -72.56486071],
        [-86.31982316, -88.04291327, -91.21567288, ..., -73.63473661,
         -76.99040949, -83.58180581]],

       [[-69.18398729, -67.41098068, -66.39033808, ..., -69.79260388,
         -77.06097386, -68.29130433],
        [         nan,          nan, -63.5556828 , ...,          nan,
         -66.99317457, -68.15231144],
        [-66.6109149 , -68.34655447, -67.58694222, ...,          nan,
         -72.91227051, -67.67163733],
...
        [         nan,          nan, -77.49739699, ...,          nan,
                  nan,          nan],
        [-75.55376669, -72.48841148, -71.831832  , ..., -71.12301185,
         -76.3407095 , -74.91035686],
        [-81.17830601, -81.80711836, -78.72356153, ..., -74.18996512,
         -79.52990337, -81.1283983 ]],

       [[-78.33982659, -80.22561847, -81.48963873, ..., -76.11904029,
         -78.15430374, -76.97820823],
        [         nan,          nan, -73.35409477, ...,          nan,
         -80.69051034, -72.56421207],
        [-78.81841249, -77.37362419, -76.62784519, ...,          nan,
         -71.30799733, -76.47833907],
        ...,
        [         nan,          nan, -78.09680427, ...,          nan,
                  nan,          nan],
        [-75.91866354, -73.13438064, -72.19797392, ..., -71.07540746,
         -76.7422887 , -75.20749839],
        [-81.21197187, -81.82743632, -78.97007954, ..., -74.15411621,
         -79.61746828, -80.92832494]]], shape=(75, 22, 12))
Coordinates:
  * range     (range) int64 600B 10 20 30 40 50 60 ... 700 710 720 730 740 750
  * province  (province) object 176B 101.0 102.0 103.0 ... 502.0 503.0 601.0
  * month     (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
Attributes:
    units:      dB re m-1
    long_name:  Volume backscattering strength

Conversion en dataframe

In [ ]:
df_sv_final= sv_mean_db.to_dataframe(name="sv_mean_db").reset_index()
df_sv_final = df_sv_final.rename(columns={"range": "profondeur", "month": "mois"})  
df_sv_final = df_sv_final[["profondeur", "province", "mois", "sv_mean_db"]]       
df_sv_final.head()    

,profondeur,province,mois,sv_mean_db
0,10,101.0,1,-71.762350
1,10,101.0,2,-69.815838
2,10,101.0,3,-69.906392
3,10,101.0,4,-70.257590
4,10,101.0,5,NaN


In [22]:
df_sv_final.to_csv("/data/rd_exchange/asauvebois/df_svMean_Plot.csv", index=False)